# Análise comparativa: Declaração do BRICS × Declaração da OCDE

**Projeto:** *As Relações Brasil-China e a Política de Governança Brasileira sobre Inteligência Artificial* (FAPESP, Processo nº 2026/04597-1)
**Etapa:** 03 (análise comparativa), conforme a skill 03 ([`skill_análise_comparativa.md`](skill_análise_comparativa.md))
**Insumos:** `BRICS_extracao.json` e `OCDE_extracao_v2.json`, produzidos pela skill 01 ([`skill_extração.md`](skill_extração.md))

---

> ⚠️ **Visualizações prévias.** Todas as visualizações geradas neste notebook (gráficos e tabelas) são **prévias** e compõem **apenas uma base para o início da construção de novos gráficos**. As novas visualizações serão construídas de acordo com os comandos e as exigências do pesquisador, que são obedecidos em última instância. Nenhuma visualização, tabela ou relatório vai para a pasta `resultados/` sem comando expresso do pesquisador.

## 1. Do que se trata

Este notebook compara, de forma **conjunta**, os textos da Declaração do BRICS e da Declaração da OCDE. Ele é a terceira etapa do fluxo de análise da pasta *Declarações IA*: a skill 01 extrai o conteúdo integral; a skill 02 analisa cada declaração isoladamente (notebooks `Análise_BRICS.ipynb` e `Análise_OCDE.ipynb`); esta etapa (skill 03) analisa os dois documentos lado a lado.

## 2. Finalidade

A finalidade deste notebook é **gerar visualizações e interpretações comparativas** sobre os dois documentos, por meio de gráficos, tabelas e medidas textuais. As análises permitem verificar:

- **semelhanças**: termos com peso relativo parecido nos dois textos;
- **distinções**: termos significativamente mais característicos de um documento do que do outro;
- **termos mais comuns** de cada documento, lado a lado.

## 3. Insumos e recorte do conteúdo

São utilizados **apenas os resultados da skill 01**, ou seja, os arquivos JSON das duas declarações gerados por aquela etapa. De cada arquivo, interessam **somente**:

- o **título** do texto (`document.title`), usado para identificar o documento;
- o **conteúdo integral e literal de cada parágrafo ou item** (`sections[].text`), que é o único conteúdo analisado.

O identificador de cada unidade (`id`) é usado apenas como índice de posição. **Todo conteúdo periférico não é utilizado**: subtítulos internos, numeração, categoria estrutural, página, lista de signatários e demais metadados ficam fora da análise.

Os dois documentos são **reprocessados aqui com o mesmo pipeline e os mesmos parâmetros**. Nenhum número é copiado dos notebooks individuais.

## 4. Por que a comparação é relativa

Os documentos têm **tamanhos diferentes**. Comparar frequências absolutas favoreceria o texto mais longo: um termo pode aparecer mais vezes em um documento apenas porque ele é maior. Por isso, **toda comparação é feita em ocorrências por 1.000 tokens**:

> **taxa por 1.000 tokens = (ocorrências do termo no documento ÷ total de tokens do documento) × 1.000**

Com essa normalização, os termos ficam em **pé de igualdade**: uma taxa de 5 por 1.000 significa o mesmo peso relativo nos dois textos, independentemente do tamanho de cada um. O denominador é o total de tokens (palavras) do documento, incluindo *stopwords*. Os valores absolutos aparecem apenas no perfil, para expor a diferença de tamanho.

Como a normalização não elimina a instabilidade de frequências baixas, as diferenças termo a termo só são consideradas **distintivas** quando passam por um teste de significância (G²) **e** por um limiar de tamanho de efeito (*Log Ratio*).

## 5. Método (visão geral)

| Etapa | Procedimento |
|---|---|
| Verificação | Integridade dos JSONs, origem na skill 01, mesmo idioma, pendências de revisão |
| Pré-processamento | Idêntico ao da skill 02 (spaCy, compostos com hífen, lemas, classes, *stopwords*) |
| Perfil comparativo | Extensão e vocabulário de cada documento; razão de tamanho |
| Termos mais comuns | Taxas por 1.000 tokens dos termos mais frequentes de cada documento, lado a lado |
| Semelhanças | Dispersão das taxas dos termos compartilhados (escala logarítmica) |
| Distinções | *Keyness*: G² (significância) + *Log Ratio* (tamanho do efeito) |
| Vocabulário | Lemas compartilhados e exclusivos (com ressalva de tamanho) |

## 6. Transparência das escolhas

**Todo método utilizado e todas as escolhas tomadas pelo pesquisador estão presentes e verificáveis** em dois lugares:

1. **no código**: os parâmetros ficam reunidos na célula de configuração, e cada função traz comentários sobre o que faz;
2. **na descrição que segue cada visualização**, composta por: *Método e escolhas*, *Como ler*, *Interpretação* e *Fragilidades e ajustes possíveis*.

As tabelas completas, as figuras e os demais arquivos **só são exportados para a pasta `resultados/` por comando expresso do pesquisador** (skill 03, seção 4.4). Sem esse comando, o notebook apenas exibe os resultados.

## 7. Princípios

Todo o conteúdo deste notebook segue os princípios da **skill 03** (análise conjunta e comparativa das declarações), que amplia a skill 02. Os dois documentos recebem tratamento **simétrico**: nenhum é tomado como referência privilegiada. A convenção de sentido das medidas assimétricas é declarada nos eixos.

> As interpretações são redigidas **somente após a execução** do notebook, com base nos valores efetivamente obtidos. Enquanto isso, os campos de interpretação ficam marcados como pendentes.

## 8. Configuração

Todos os parâmetros editáveis estão nesta célula. Os parâmetros de pré-processamento são os mesmos dos notebooks individuais. A gravação em `resultados/` está desativada (`SALVAR_SAIDAS = False`) e só é ativada por comando do pesquisador.

In [ ]:
import hashlib
import json
import math
import platform
import re
import unicodedata
import zlib
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy
from IPython.display import Markdown, display
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, MaxNLocator

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 140)

# =====================================================================
# CONFIGURAÇÃO — parâmetros editáveis pelo pesquisador
# =====================================================================
# Documentos comparados (a ordem define a ordem nas figuras e o sentido do Log Ratio).
DOCUMENTOS = {
    "BRICS": "BRICS_extracao.json",   # entregável da skill 01
    "OCDE": "OCDE_extracao_v2.json",   # entregável da skill 01 (versão v2)
}
ARQUIVO_CODEBOOK = "codebook_declaracoes.json"

# Saídas versionadas (resultados anteriores nunca são sobrescritos).
NOME_SAIDA = "Comparativa_" + "_".join(DOCUMENTOS)
VERSAO_SAIDA = "v1"
# Gravação em resultados/ DESATIVADA: visualizações, tabelas e relatórios só vão para
# resultados/ por comando expresso do pesquisador (skill, seção 4.4).
SALVAR_SAIDAS = False
SOBRESCREVER = False

# Recorte do conteúdo
INCLUIR_TITULO_NA_CONTAGEM = False

# Pré-processamento (idêntico aos notebooks individuais)
IDIOMA_ESPERADO = "en"
MODELO_SPACY = "en_core_web_sm"
CLASSES_DE_CONTEUDO = {"NOUN", "PROPN", "VERB", "ADJ", "ADV"}
TAMANHO_MINIMO_LEMA = 2
STOPWORDS_ADICIONAIS = []
STOPWORDS_MANTIDAS = []
CORRECOES_DE_LEMA = {
    "datum": "data",
}

# Comparação termo a termo
TOP_N_COMPARACAO = 15           # termos mais frequentes de CADA documento no gráfico lado a lado
FREQ_MINIMA_COMPARACAO = 5      # frequência combinada mínima para dispersão e keyness
LIMIAR_G2 = 6.63                # G² crítico para p < 0,01 (1 grau de liberdade)
LIMIAR_LOG_RATIO = 1.0          # |Log Ratio| >= 1: ao menos o dobro da frequência relativa
TOP_N_KEYNESS = 15              # termos distintivos exibidos por documento
ALFA_BONFERRONI = 0.01          # nível usado no relato da correção para comparações múltiplas
N_ROTULOS_DISPERSAO = 12        # termos rotulados no gráfico de semelhanças

# Incerteza (bootstrap por parágrafos/itens)
N_BOOTSTRAP = 2000
NIVEL_CONFIANCA = 0.95
SEMENTE = 2026

## 9. Ambiente, estilo e funções

As funções do pipeline são **idênticas** às dos notebooks individuais; as funções comparativas vêm em seguida.

In [ ]:
# ---------------------------------------------------------------------------
# Restrição de ambiente: o notebook só roda de dentro da pasta "Declarações IA".
# ---------------------------------------------------------------------------
PASTA = Path.cwd()
if unicodedata.normalize("NFC", PASTA.name) != "Declarações IA":
    raise RuntimeError(
        "Este notebook deve ser executado de dentro da pasta 'Declarações IA' "
        f"(restrição de ambiente da skill). Pasta atual: {PASTA}"
    )

# ---------------------------------------------------------------------------
# Pasta de saída versionada: resultados anteriores nunca são sobrescritos.
# ---------------------------------------------------------------------------
DIR_SAIDA = PASTA / "resultados" / NOME_SAIDA / VERSAO_SAIDA
if SALVAR_SAIDAS:
    if DIR_SAIDA.exists() and any(DIR_SAIDA.iterdir()) and not SOBRESCREVER:
        raise FileExistsError(
            f"A pasta de saída '{DIR_SAIDA.relative_to(PASTA)}' já contém arquivos. "
            "Para preservar a versão anterior, altere VERSAO_SAIDA (ex.: 'v2') na configuração. "
            "Use SOBRESCREVER = True somente se a substituição for intencional."
        )
    DIR_SAIDA.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Estilo das figuras (padrão acadêmico: fonte >= 9 pt, 300 dpi, PNG + PDF)
# ---------------------------------------------------------------------------
# Cores fixas por documento, iguais em todos os notebooks da pasta.
# O par laranja/azul foi verificado para leitores com daltonismo.
COR_DOCUMENTO = {"BRICS": "#eb6834", "OCDE": "#2a78d6"}
COR_TINTA = "#0b0b0b"     # texto principal
COR_TINTA_2 = "#52514e"   # texto secundário (rótulos, anotações)
COR_GRADE = "#e4e3df"     # grade e linhas de apoio
COR_LIGACAO = "#b9b8b2"   # segmentos de ligação (gráficos de halteres)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": COR_TINTA_2,
    "axes.linewidth": 0.6,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": COR_GRADE,
    "grid.linewidth": 0.6,
    "grid.linestyle": "-",
    "text.color": COR_TINTA,
    "axes.labelcolor": COR_TINTA,
    "xtick.color": COR_TINTA_2,
    "ytick.color": COR_TINTA_2,
    "legend.frameon": False,
    "pdf.fonttype": 42,
})

# ---------------------------------------------------------------------------
# Funções auxiliares de formatação e gravação
# ---------------------------------------------------------------------------
def num(x, casas=2):
    """Formata número no padrão brasileiro (vírgula decimal, ponto de milhar)."""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "—"
    texto = f"{x:,.{casas}f}"
    return texto.replace(",", "§").replace(".", ",").replace("§", ".")


# Rótulos de eixo com vírgula decimal.
FMT_BR = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))


def exibir(df, casas=2):
    """Exibe uma tabela com vírgula decimal, sem alterar os valores."""
    display(df.style.format(precision=casas, decimal=",", thousands=".", na_rep="—"))


def salvar_figura(fig, nome):
    """Grava a figura em PNG (300 dpi) e PDF na pasta de saída versionada."""
    if SALVAR_SAIDAS:
        for ext in ("png", "pdf"):
            fig.savefig(DIR_SAIDA / f"{nome}.{ext}")


def salvar_tabela(df, nome, index=True):
    """Grava a tabela completa em CSV (UTF-8 com BOM, ';', vírgula decimal)."""
    if SALVAR_SAIDAS:
        df.to_csv(DIR_SAIDA / f"{nome}.csv", sep=";", decimal=",", encoding="utf-8-sig", index=index)


def semente_documento(sigla):
    """Semente do bootstrap própria de cada documento, derivada de SEMENTE e da sigla.
    Garante (a) os mesmos intervalos no notebook individual e no comparativo e
    (b) sequências aleatórias independentes entre documentos diferentes."""
    return np.random.SeedSequence([SEMENTE, zlib.crc32(sigla.encode("utf-8"))])


print(f"Pasta de trabalho: {PASTA}")
print(f"Saídas: {DIR_SAIDA.relative_to(PASTA) if SALVAR_SAIDAS else 'gravação desativada'}")

SIGLAS = list(DOCUMENTOS)
if len(SIGLAS) != 2:
    raise ValueError("Este notebook compara exatamente dois documentos.")
SIG_A, SIG_B = SIGLAS
COR = {s: COR_DOCUMENTO.get(s, cor) for s, cor in zip(SIGLAS, ["#eb6834", "#2a78d6"])}

In [ ]:
# ===========================================================================
# PIPELINE DE PRÉ-PROCESSAMENTO E MEDIDAS
# (idêntico em todos os notebooks de análise da pasta "Declarações IA")
# ===========================================================================

# Token (palavra) = sequência de letras, admitindo hífen interno (multi-stakeholder)
# ou sigla com pontos (U.S.). Números, pontuação e símbolos NÃO são tokens.
RE_PALAVRA = re.compile(r"(?:[^\W\d_]\.){2,}|[^\W\d_]+(?:[-‐‑][^\W\d_]+)*")
HIFENS = {"-", "‐", "‑"}


def normalizar_hifen(texto):
    """Uniformiza variantes tipográficas do hífen (apenas nas chaves de contagem;
    o texto do JSON não é alterado)."""
    return texto.replace("‐", "-").replace("‑", "-")


def carregar_extracao(caminho):
    """Lê o JSON produzido pela skill 01 e devolve SOMENTE o título e as unidades
    (id, texto integral). Todo o conteúdo periférico é descartado aqui.
    Interrompe a execução se o arquivo não passar nas verificações de integridade."""
    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo '{caminho.name}' não encontrado na pasta. "
            "Execute antes a skill 01 (skill_extração.md) para gerar o JSON da declaração."
        )
    bruto = caminho.read_bytes()
    dados = json.loads(bruto.decode("utf-8"))

    problemas = [f"chave '{c}' ausente" for c in ("document", "extraction", "sections") if c not in dados]
    if problemas:
        raise ValueError("Falha na verificação do JSON:\n- " + "\n- ".join(problemas))

    extracao = dados["extraction"]
    if extracao.get("skill") != "skill_extração":
        problemas.append("o JSON não registra ter sido produzido pela skill 01 (extraction.skill)")
    secoes = dados["sections"]
    if [s.get("id") for s in secoes] != list(range(1, len(secoes) + 1)):
        problemas.append("os ids das unidades não são sequenciais a partir de 1")
    vazias = [s.get("id") for s in secoes if not isinstance(s.get("text"), str) or not s["text"].strip()]
    if vazias:
        problemas.append(f"unidades com texto vazio: {vazias}")
    if extracao.get("units_extracted") is not None and extracao["units_extracted"] != len(secoes):
        problemas.append(
            f"extraction.units_extracted = {extracao['units_extracted']}, mas há {len(secoes)} unidades"
        )
    if not dados["document"].get("title"):
        problemas.append("título (document.title) ausente")
    if problemas:
        raise ValueError("Falha na verificação do JSON:\n- " + "\n- ".join(problemas))

    return {
        "arquivo": caminho.name,
        "sha256": hashlib.sha256(bruto).hexdigest(),
        "titulo": dados["document"]["title"],
        "idioma": dados["document"].get("language"),
        "pendencias_revisao": extracao.get("items_flagged_for_review"),
        "unidades": [(s["id"], s["text"]) for s in secoes],   # SOMENTE id (posição) e texto
    }


def montar_stopwords(nlp):
    """Lista de stopwords do spaCy, ajustada pelas listas editáveis da configuração."""
    base = {w.lower() for w in nlp.Defaults.stop_words}
    return (base | {w.lower() for w in STOPWORDS_ADICIONAIS}) - {w.lower() for w in STOPWORDS_MANTIDAS}


def unir_compostos_hifenizados(doc):
    """Reúne em um único token as palavras compostas com hífen (ex.: multi-stakeholder,
    human-centric, state-of-the-art), que o spaCy separa em vários tokens.
    Só une quando não há espaço em volta do hífen."""
    trechos, i, n = [], 0, len(doc)
    while i < n:
        j = i
        while (j + 2 < n and doc[j].is_alpha and doc[j].whitespace_ == ""
               and doc[j + 1].text in HIFENS and doc[j + 1].whitespace_ == ""
               and doc[j + 2].is_alpha):
            j += 2
        if j > i:
            trechos.append(doc[i:j + 1])
            i = j + 1
        else:
            i += 1
    with doc.retokenize() as rt:
        for trecho in trechos:
            ultimo = trecho[-1]
            # Lema do composto = forma escrita; se o último elemento estiver no plural,
            # usa-se o lema dele (multi-stakeholders -> multi-stakeholder).
            if ultimo.tag_ in {"NNS", "NNPS"}:
                lema = trecho.text[: len(trecho.text) - len(ultimo.text)] + ultimo.lemma_
            else:
                lema = trecho.text
            rt.merge(trecho, attrs={"LEMMA": lema, "POS": ultimo.pos_, "TAG": ultimo.tag_})
    return doc


def processar_documento(unidades, nlp, stopwords):
    """Aplica o pré-processamento a cada unidade (parágrafo/item) e devolve uma tabela
    com uma linha por token (palavra), na ordem original do documento."""
    linhas, posicao = [], 0
    for (uid, _), doc in zip(unidades, nlp.pipe([texto for _, texto in unidades])):
        doc = unir_compostos_hifenizados(doc)
        for k, frase in enumerate(doc.sents):
            for tok in frase:
                if not RE_PALAVRA.fullmatch(tok.text):
                    continue
                lema = normalizar_hifen(tok.lemma_.lower())
                lema = CORRECOES_DE_LEMA.get(lema, lema)
                linhas.append({
                    "unidade": uid,
                    "frase": f"{uid}.{k}",
                    "i": tok.i,                      # índice no parágrafo (detecta adjacência)
                    "posicao": posicao,              # posição no documento inteiro
                    "forma": normalizar_hifen(tok.text.lower()),
                    "lema": lema,
                    "classe": tok.pos_,
                    "etiqueta": tok.tag_,
                })
                posicao += 1
    tokens = pd.DataFrame(linhas)
    tokens["conteudo"] = (
        tokens["classe"].isin(CLASSES_DE_CONTEUDO)
        & ~tokens["lema"].isin(stopwords)
        & (tokens["lema"].str.len() >= TAMANHO_MINIMO_LEMA)
        & tokens["lema"].map(lambda lema: bool(RE_PALAVRA.fullmatch(lema)))
    )
    return tokens


def perfil_documento(tokens, n_unidades, rotulo):
    """Indicadores descritivos de extensão e vocabulário."""
    conteudo = tokens[tokens["conteudo"]]
    tam = tokens.groupby("unidade").size()
    freq_cont = conteudo["lema"].value_counts()
    return pd.Series({
        "Unidades (parágrafos/itens)": n_unidades,
        "Unidades sem nenhum token": n_unidades - tokens["unidade"].nunique(),
        "Tokens (palavras)": len(tokens),
        "Formas distintas": tokens["forma"].nunique(),
        "Tokens de conteúdo": len(conteudo),
        "Tokens de conteúdo (% dos tokens)": len(conteudo) / len(tokens) * 100,
        "Lemas de conteúdo distintos": conteudo["lema"].nunique(),
        "Lemas de conteúdo com uma única ocorrência": int((freq_cont == 1).sum()),
        "Tokens por unidade: média": tam.mean(),
        "Tokens por unidade: mediana": tam.median(),
        "Tokens por unidade: mínimo": tam.min(),
        "Tokens por unidade: máximo": tam.max(),
    }, name=rotulo)


def tabela_frequencias(tokens):
    """Frequência, taxa por 1.000 tokens, alcance e dispersão (DP normalizado) de
    TODOS os lemas de conteúdo. Ordenação: frequência decrescente; empate -> ordem alfabética."""
    n_tokens = len(tokens)
    unidades = sorted(tokens["unidade"].unique())
    proporcao_unidade = (tokens.groupby("unidade").size().reindex(unidades) / n_tokens).to_numpy()
    conteudo = tokens[tokens["conteudo"]]
    cruz = pd.crosstab(conteudo["lema"], conteudo["unidade"]).reindex(columns=unidades, fill_value=0)
    freq = cruz.sum(axis=1)
    # DP de Gries (2008), normalizado por Lijffijt & Gries (2012):
    # 0 = distribuição proporcional ao tamanho das unidades; 1 = concentração máxima.
    v = cruz.div(freq, axis=0).to_numpy()
    dp = 0.5 * np.abs(v - proporcao_unidade).sum(axis=1)
    dp_norm = dp / (1 - proporcao_unidade.min()) if len(unidades) > 1 else np.full(len(freq), np.nan)
    formas = conteudo.groupby("lema")["forma"].agg(
        lambda s: "; ".join(f"{forma} ({n})" for forma, n in s.value_counts().items())
    )
    tab = pd.DataFrame({
        "frequencia": freq,
        "por_1000_tokens": freq / n_tokens * 1000,
        "unidades_com_ocorrencia": (cruz > 0).sum(axis=1),
        "pct_unidades": (cruz > 0).sum(axis=1) / len(unidades) * 100,
        "DP_norm": dp_norm,
        "formas_observadas": formas,
    })
    tab.index.name = "lema"
    tab = tab.reset_index().sort_values(["frequencia", "lema"], ascending=[False, True])
    return tab.set_index("lema")


def relatar_empates(tab, coluna, n, rotulo):
    """Informa termos empatados no ponto de corte de um recorte 'top N'."""
    if len(tab) <= n:
        return
    corte = tab[coluna].iloc[n - 1]
    fora = tab.iloc[n:]
    fora = fora[fora[coluna] == corte]
    if len(fora):
        lista = ", ".join(fora.index[:20]) + (" …" if len(fora) > 20 else "")
        print(
            f"Empate no ponto de corte: o {n}º {rotulo} tem valor {num(corte, 2)}; "
            f"outros {len(fora)} com o mesmo valor ficaram fora do recorte "
            f"(desempate por ordem alfabética): {lista}"
        )


def tabela_bigramas(tokens):
    """Pares de tokens de conteúdo ADJACENTES no texto original (mesma frase, sem
    pontuação entre eles). Não se formam pares artificiais pela remoção de stopwords."""
    a = tokens.iloc[:-1].reset_index(drop=True)
    b = tokens.iloc[1:].reset_index(drop=True)
    ok = (
        (a["frase"].to_numpy() == b["frase"].to_numpy())
        & (b["i"].to_numpy() - a["i"].to_numpy() == 1)
        & a["conteudo"].to_numpy() & b["conteudo"].to_numpy()
    )
    pares = pd.DataFrame({
        "bigrama": (a["lema"] + " " + b["lema"])[ok],
        "forma": (a["forma"] + " " + b["forma"])[ok],
        "unidade": a["unidade"][ok],
    })
    if pares.empty:
        return pd.DataFrame(columns=["frequencia", "por_1000_tokens", "unidades_com_ocorrencia",
                                     "forma_principal", "formas_observadas"]).rename_axis("bigrama")
    grupos = pares.groupby("bigrama")
    tab = pd.DataFrame({
        "frequencia": grupos.size(),
        "por_1000_tokens": grupos.size() / len(tokens) * 1000,
        "unidades_com_ocorrencia": grupos["unidade"].nunique(),
        "forma_principal": grupos["forma"].agg(lambda s: s.value_counts().index[0]),
        "formas_observadas": grupos["forma"].agg(
            lambda s: "; ".join(f"{f} ({n})" for f, n in s.value_counts().items())
        ),
    })
    tab = tab.reset_index().sort_values(["frequencia", "bigrama"], ascending=[False, True])
    return tab.set_index("bigrama")


def carregar_codebook(caminho):
    """Lê o codebook e verifica a exclusividade mútua das categorias.
    Devolve o codebook e a lista de entradas ordenada da expressão mais longa para a mais curta."""
    codebook = json.loads(caminho.read_text(encoding="utf-8"))
    entradas, vistos = [], {}
    for categoria, info in codebook["categorias"].items():
        for termo in info["termos"]:
            chave = tuple(normalizar_hifen(termo.lower()).split())
            if chave in vistos:
                raise ValueError(
                    f"Termo '{termo}' aparece em '{vistos[chave]}' e em '{categoria}'. "
                    "As categorias do codebook devem ser mutuamente exclusivas."
                )
            vistos[chave] = categoria
            entradas.append((chave, categoria, termo))
    entradas.sort(key=lambda e: -len(e[0]))
    sha = hashlib.sha256(caminho.read_bytes()).hexdigest()
    return codebook, entradas, sha


def aplicar_codebook(tokens, entradas):
    """Localiza os termos do codebook no texto. Cada palavra do termo é comparada com a
    forma OU com o lema do token. Expressões exigem palavras contíguas na mesma frase.
    Prevalece a expressão mais longa, e cada token é contado no máximo uma vez."""
    achados = []
    for _, frase in tokens.groupby("frase", sort=False):
        formas = frase["forma"].tolist()
        lemas = frase["lema"].tolist()
        idx = frase["i"].tolist()
        pos = frase["posicao"].tolist()
        uid = frase["unidade"].iloc[0]
        j, n = 0, len(formas)
        while j < n:
            for chave, categoria, termo in entradas:
                tam = len(chave)
                if j + tam > n:
                    continue
                if all(formas[j + k] == chave[k] or lemas[j + k] == chave[k] for k in range(tam)) and \
                   all(idx[j + k + 1] - idx[j + k] == 1 for k in range(tam - 1)):
                    achados.append({"unidade": uid, "posicao": pos[j], "categoria": categoria,
                                    "termo": termo, "forma": " ".join(formas[j:j + tam]), "n_tokens": tam})
                    j += tam
                    break
            else:
                j += 1
    return pd.DataFrame(achados, columns=["unidade", "posicao", "categoria", "termo", "forma", "n_tokens"])


def taxas_bootstrap(contagens, tamanhos, semente):
    """Bootstrap por unidades (parágrafos/itens): reamostra unidades com reposição e
    recalcula a taxa por 1.000 tokens. Respeita a dependência entre palavras de um
    mesmo parágrafo. contagens: matriz (rótulos x unidades). Devolve (rótulos x reamostras)."""
    rng = np.random.default_rng(semente)
    n_u = len(tamanhos)
    idx = rng.integers(0, n_u, size=(N_BOOTSTRAP, n_u))
    return contagens[:, idx].sum(axis=2) / tamanhos[idx].sum(axis=1) * 1000


def tabela_categorias(achados, tokens, categorias, semente):
    """Ocorrências por categoria, taxa por 1.000 tokens com IC por bootstrap,
    participação no total do codebook e alcance."""
    unidades = sorted(tokens["unidade"].unique())
    tamanhos = tokens.groupby("unidade").size().reindex(unidades).to_numpy()
    if achados.empty:
        matriz = pd.DataFrame(0, index=categorias, columns=unidades)
    else:
        matriz = pd.crosstab(achados["categoria"], achados["unidade"]).reindex(
            index=categorias, columns=unidades, fill_value=0)
    reamostras = taxas_bootstrap(matriz.to_numpy(), tamanhos, semente)
    alfa = (1 - NIVEL_CONFIANCA) / 2
    ocorr = matriz.sum(axis=1)
    tab = pd.DataFrame({
        "ocorrencias": ocorr,
        "por_1000_tokens": ocorr / len(tokens) * 1000,
        "ic_inferior": np.quantile(reamostras, alfa, axis=1),
        "ic_superior": np.quantile(reamostras, 1 - alfa, axis=1),
        "pct_do_codebook": ocorr / ocorr.sum() * 100 if ocorr.sum() else np.nan,
        "unidades_com_ocorrencia": (matriz > 0).sum(axis=1),
    })
    tab.index.name = "categoria"
    return tab, reamostras


def tabela_termos_codebook(achados, entradas, n_tokens):
    """Contribuição de cada termo do codebook, incluindo os termos sem ocorrência."""
    linhas = []
    for _, categoria, termo in sorted(entradas, key=lambda e: (list(CATEGORIAS).index(e[1]), e[2])):
        sub = achados[(achados["categoria"] == categoria) & (achados["termo"] == termo)]
        linhas.append({
            "categoria": categoria,
            "termo": termo,
            "ocorrencias": len(sub),
            "por_1000_tokens": len(sub) / n_tokens * 1000,
            "formas_observadas": "; ".join(f"{f} ({n})" for f, n in sub["forma"].value_counts().items()),
        })
    tab = pd.DataFrame(linhas)
    total_cat = tab.groupby("categoria")["ocorrencias"].transform("sum")
    tab["pct_da_categoria"] = np.where(total_cat > 0, tab["ocorrencias"] / total_cat.where(total_cat > 0, 1) * 100, np.nan)
    return tab


def robustez_termo_dominante(termos, n_tokens):
    """Teste de robustez: quanto cada categoria depende do seu termo mais frequente."""
    linhas = []
    for categoria in CATEGORIAS:
        sub = termos[termos["categoria"] == categoria].sort_values(
            ["ocorrencias", "termo"], ascending=[False, True])
        total = sub["ocorrencias"].sum()
        dominante = sub.iloc[0]
        linhas.append({
            "categoria": categoria,
            "termo_dominante": dominante["termo"] if total else "—",
            "ocorrencias_do_termo": dominante["ocorrencias"] if total else 0,
            "participacao_do_termo_pct": dominante["ocorrencias"] / total * 100 if total else np.nan,
            "por_1000_com_o_termo": total / n_tokens * 1000,
            "por_1000_sem_o_termo": (total - dominante["ocorrencias"]) / n_tokens * 1000,
        })
    tab = pd.DataFrame(linhas).set_index("categoria")
    tab["alerta"] = np.where(tab["participacao_do_termo_pct"] > 50,
                             "mais da metade da categoria vem de um único termo", "")
    return tab


MODAIS = ["must", "shall", "should", "will", "would", "may", "might", "can", "could"]


def tabela_modais(tokens):
    """Verbos modais (etiqueta gramatical MD), por lema, por 1.000 tokens.
    Ordem fixa; modais fora da lista padrão, se houver, entram ao final."""
    modais = tokens[tokens["etiqueta"] == "MD"]["lema"].value_counts()
    ordem = MODAIS + sorted(set(modais.index) - set(MODAIS))
    modais = modais.reindex(ordem, fill_value=0)
    return pd.DataFrame({"ocorrencias": modais, "por_1000_tokens": modais / len(tokens) * 1000}).rename_axis("modal")


def grafico_dispersao_posicional(ax, posicoes_por_linha, rotulos, inicios_unidades, cor):
    """Gráfico de dispersão posicional: cada traço é uma ocorrência, na posição
    relativa em que aparece no documento (0 = início; 1 = fim)."""
    if len(inicios_unidades) <= 120:
        for x in inicios_unidades:
            ax.axvline(x, color=COR_GRADE, lw=0.5, zorder=0)
    for y, posicoes in enumerate(posicoes_por_linha):
        ax.vlines(posicoes, y - 0.36, y + 0.36, color=cor, lw=0.9)
    ax.set_yticks(range(len(rotulos)), labels=rotulos)
    ax.set_ylim(len(rotulos) - 0.5, -0.5)
    ax.set_xlim(0, 1)
    ax.grid(False)
    ax.xaxis.set_major_formatter(FMT_BR)
    ax.set_xlabel("Posição relativa no documento (0 = início; 1 = fim)")


def versoes_ambiente(nlp):
    return {
        "python": platform.python_version(),
        "spacy": spacy.__version__,
        "modelo_spacy": f"{MODELO_SPACY} {nlp.meta.get('version', '?')}",
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "matplotlib": matplotlib.__version__,
    }

In [ ]:
# ===========================================================================
# FUNÇÕES COMPARATIVAS (skill 03)
# ===========================================================================
from scipy.stats import chi2


def tabela_keyness(freq_a, freq_b, n_a, n_b, sig_a, sig_b):
    """Compara todos os lemas de conteúdo dos dois documentos.
    G² (log-verossimilhança; Dunning, 1993; Rayson & Garside, 2000) mede a significância;
    Log Ratio (Hardie, 2014) mede o tamanho do efeito: log2 da razão entre as taxas relativas.
    Convenção: Log Ratio > 0 -> mais característico de sig_a; < 0 -> de sig_b.
    Frequência zero recebe correção de 0,5 apenas no cálculo do Log Ratio."""
    lemas = freq_a.index.union(freq_b.index)
    a = freq_a["frequencia"].reindex(lemas, fill_value=0).astype(float)
    b = freq_b["frequencia"].reindex(lemas, fill_value=0).astype(float)
    esp_a = n_a * (a + b) / (n_a + n_b)
    esp_b = n_b * (a + b) / (n_a + n_b)
    with np.errstate(divide="ignore", invalid="ignore"):
        g2 = 2 * (np.where(a > 0, a * np.log(a / esp_a), 0.0) + np.where(b > 0, b * np.log(b / esp_b), 0.0))
    log_ratio = np.log2((a.where(a > 0, 0.5) / n_a) / (b.where(b > 0, 0.5) / n_b))
    tab = pd.DataFrame({
        f"freq_{sig_a}": a.astype(int),
        f"freq_{sig_b}": b.astype(int),
        f"por_1000_{sig_a}": a / n_a * 1000,
        f"por_1000_{sig_b}": b / n_b * 1000,
        "freq_combinada": (a + b).astype(int),
        "G2": g2,
        "p_valor": chi2.sf(g2, 1),
        "log_ratio": log_ratio,
    })
    tab["presenca"] = np.select([(a > 0) & (b > 0), a > 0], ["ambos", f"só {sig_a}"], f"só {sig_b}")
    tab.index.name = "lema"
    return tab.sort_values(["G2", "freq_combinada"], ascending=[False, False])

## 10. Carregamento e verificação dos insumos

Cada JSON passa pelas mesmas verificações dos notebooks individuais. Além disso, os dois documentos precisam estar no mesmo idioma, já que documentos em idiomas diferentes não são comparados lexicalmente.

In [ ]:
documentos = {s: carregar_extracao(PASTA / arquivo) for s, arquivo in DOCUMENTOS.items()}
idiomas = {s: d["idioma"] for s, d in documentos.items()}
if len(set(idiomas.values())) != 1 or IDIOMA_ESPERADO not in idiomas.values():
    raise ValueError(f"Idiomas dos documentos: {idiomas}. A comparação exige o mesmo idioma ({IDIOMA_ESPERADO!r}).")

unidades = {}
for s, d in documentos.items():
    unidades[s] = ([(0, d["titulo"])] if INCLUIR_TITULO_NA_CONTAGEM else []) + list(d["unidades"])
    print(f"[{s}] {d['titulo']}")
    print(f"      arquivo: {d['arquivo']} · SHA-256: {d['sha256']}")
    print(f"      unidades: {len(d['unidades'])} · idioma: {d['idioma']}")
    if d["pendencias_revisao"]:
        print(f"      ATENÇÃO: {d['pendencias_revisao']} item(ns) pendente(s) de revisão humana no CSV da skill 01.")

## 11. Pré-processamento e perfil comparativo

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
nlp = spacy.load(MODELO_SPACY)
STOPWORDS = montar_stopwords(nlp)
tokens = {s: processar_documento(unidades[s], nlp, STOPWORDS) for s in SIGLAS}
N = {s: len(tokens[s]) for s in SIGLAS}
freqs = {s: tabela_frequencias(tokens[s]) for s in SIGLAS}

perfil = pd.concat([perfil_documento(tokens[s], len(unidades[s]), s) for s in SIGLAS], axis=1)
exibir(perfil)
salvar_tabela(perfil, "00_perfil_comparativo")

maior, menor = max(SIGLAS, key=N.get), min(SIGLAS, key=N.get)
print(f"Síntese factual: {maior} tem {N[maior]} tokens e {menor} tem {N[menor]}; "
      f"o documento {maior} é {num(N[maior] / N[menor], 2)} vezes maior. "
      "Por isso, todas as comparações a seguir usam ocorrências por 1.000 tokens.")

**Método e escolhas.** Os dois documentos passam pelo mesmo pré-processamento, com os mesmos parâmetros. O perfil traz valores **absolutos** apenas para **expor a disparidade de tamanho**, que justifica a normalização relativa adotada em todas as seções seguintes.

**Como ler.** A linha *Tokens (palavras)* é o denominador das taxas por 1.000 tokens de cada documento. A razão de tamanho, informada abaixo da tabela, mostra quanto as frequências absolutas seriam distorcidas se fossem comparadas diretamente.

**Fragilidades e ajustes possíveis.** Indicadores de vocabulário (formas distintas, lemas distintos, lemas com uma única ocorrência) **crescem com a extensão do texto** e não medem riqueza vocabular de forma comparável entre documentos de tamanhos diferentes. Por isso, não são usados para conclusões comparativas.

## 12. Termos mais comuns, lado a lado

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
taxas = pd.DataFrame({s: freqs[s]["por_1000_tokens"] for s in SIGLAS}).fillna(0.0)
for s in SIGLAS:
    relatar_empates(freqs[s], "frequencia", TOP_N_COMPARACAO, f"lema de {s}")
uniao = list(dict.fromkeys(freqs[SIG_A].head(TOP_N_COMPARACAO).index.tolist()
                           + freqs[SIG_B].head(TOP_N_COMPARACAO).index.tolist()))
sub = taxas.loc[uniao]
sub = sub.loc[sub.max(axis=1).sort_values(ascending=False, kind="mergesort").index]

fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(sub) + 1.5))
y = np.arange(len(sub))
ax.hlines(y, sub.min(axis=1), sub.max(axis=1), color=COR_LIGACAO, lw=1.5, zorder=1)
for s in SIGLAS:
    ax.scatter(sub[s], y, s=40, color=COR[s], zorder=3, label=s, edgecolors="white", linewidths=1)
ax.set_yticks(y, labels=sub.index)
ax.set_ylim(len(sub) - 0.5, -0.5)
ax.set_xlim(0, sub.to_numpy().max() * 1.08)
ax.xaxis.set_major_formatter(FMT_BR)
ax.set_xlabel("Ocorrências por 1.000 tokens")
ax.set_title(f"Termos mais frequentes de cada documento ({TOP_N_COMPARACAO} de cada, união)", loc="left")
ax.legend(loc="lower right")
ax.grid(axis="y", visible=False)
salvar_figura(fig, "01_termos_mais_comuns")
plt.show()

keyness = tabela_keyness(freqs[SIG_A], freqs[SIG_B], N[SIG_A], N[SIG_B], SIG_A, SIG_B)
salvar_tabela(keyness, "03_keyness_completa")
exibir(keyness.loc[sub.index, [f"freq_{SIG_A}", f"por_1000_{SIG_A}", f"freq_{SIG_B}", f"por_1000_{SIG_B}", "G2", "log_ratio"]])
comuns = set(freqs[SIG_A].head(TOP_N_COMPARACAO).index) & set(freqs[SIG_B].head(TOP_N_COMPARACAO).index)
print(f"Síntese factual: {len(uniao)} lemas na união; {len(comuns)} estão entre os {TOP_N_COMPARACAO} "
      f"mais frequentes dos dois documentos: {', '.join(sorted(comuns)) or '(nenhum)'}.")

**Método e escolhas.** União dos *N* lemas de conteúdo mais frequentes de cada documento (*N* = `TOP_N_COMPARACAO`), com a taxa por 1.000 tokens de cada um. Cada linha liga os dois valores do mesmo lema. Ordenação pela maior das duas taxas. Lemas ausentes em um documento aparecem com taxa zero. A tabela traz as frequências absolutas, as taxas, o G² e o *Log Ratio* de cada lema.

**Como ler.** Pontos próximos indicam peso relativo semelhante nos dois textos; linhas longas indicam diferença de ênfase. A cor identifica o documento (a mesma em todos os notebooks).

**Interpretação.** *[Pendente. A redigir somente após a execução do notebook com os JSONs definitivos da skill 01, exclusivamente a partir dos valores exibidos acima. Toda afirmação de diferença deve indicar o critério que a sustenta (skill 03, seção 4.3).]*

**Fragilidades e ajustes possíveis.** A distância visual entre os pontos **não** é, por si, um teste. Diferenças só devem ser afirmadas quando confirmadas pelo G² e pelo *Log Ratio* (seção 14). O recorte depende do parâmetro *N*, e empates no corte são informados acima do gráfico.

## 13. Semelhanças: termos compartilhados

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
compart = keyness[(keyness["presenca"] == "ambos") & (keyness["freq_combinada"] >= FREQ_MINIMA_COMPARACAO)].copy()
xa, xb = compart[f"por_1000_{SIG_A}"], compart[f"por_1000_{SIG_B}"]

fig, ax = plt.subplots(figsize=(6.8, 6.4))
lim_inf = min(xa.min(), xb.min()) / 1.6
lim_sup = max(xa.max(), xb.max()) * 1.6
grade = np.array([lim_inf, lim_sup])
ax.plot(grade, grade, color=COR_TINTA_2, lw=0.9, zorder=1)
ax.plot(grade, grade * 2, color=COR_GRADE, lw=0.9, zorder=1)
ax.plot(grade, grade / 2, color=COR_GRADE, lw=0.9, zorder=1)
ax.scatter(xa, xb, s=30, color=COR_TINTA_2, alpha=0.55, edgecolors="white", linewidths=0.8, zorder=2)
# Rótulos: lemas de maior taxa somada. Lemas com coordenadas idênticas (pontos sobrepostos)
# recebem um único rótulo conjunto, para que nenhum fique oculto.
rotulados = (xa + xb).sort_values(ascending=False, kind="mergesort").head(N_ROTULOS_DISPERSAO).index
grupos = {}
for lema in rotulados:
    grupos.setdefault((round(xa[lema], 9), round(xb[lema], 9)), []).append(lema)
anotacoes = [ax.annotate(", ".join(nomes), (gx, gy), xytext=(5, 3), textcoords="offset points",
                         fontsize=9, color=COR_TINTA) for (gx, gy), nomes in grupos.items()]
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(lim_inf, lim_sup)
ax.set_ylim(lim_inf, lim_sup)
ax.set_aspect("equal")
# Marcas do eixo em valores 1-2-5 (0,5; 1; 2; 5; 10; 20 ...), sem notação científica.
expoentes = range(int(np.floor(np.log10(lim_inf))), int(np.ceil(np.log10(lim_sup))) + 1)
marcas = [m * 10.0 ** e for e in expoentes for m in (1, 2, 5) if lim_inf <= m * 10.0 ** e <= lim_sup]
for eixo in (ax.xaxis, ax.yaxis):
    eixo.set_major_locator(matplotlib.ticker.FixedLocator(marcas))
    eixo.set_major_formatter(FMT_BR)
    eixo.set_minor_formatter(matplotlib.ticker.NullFormatter())
# Evita rótulos sobrepostos: desloca para baixo, em passos de 11 pt, o rótulo que colidir
# com um já posicionado (os pontos não se movem; só o texto).
fig.canvas.draw()
posicionadas = []
for anot in anotacoes:
    for _ in range(12):
        caixa = anot.get_window_extent()
        if not any(caixa.overlaps(outra) for outra in posicionadas):
            break
        dx, dy = anot.xyann
        anot.xyann = (dx, dy - 11)
        fig.canvas.draw()
    posicionadas.append(anot.get_window_extent())
ax.set_xlabel(f"{SIG_A}: ocorrências por 1.000 tokens (escala log)")
ax.set_ylabel(f"{SIG_B}: ocorrências por 1.000 tokens (escala log)")
ax.set_title("Termos presentes nos dois documentos", loc="left")
ax.text(0.02, 0.98, f"acima da diagonal: mais frequente em {SIG_B}\nabaixo: mais frequente em {SIG_A}\n"
        "linhas claras: o dobro / a metade", transform=ax.transAxes, va="top", fontsize=9, color=COR_TINTA_2)
salvar_figura(fig, "02_semelhancas_dispersao")
plt.show()

razao = np.abs(compart["log_ratio"])
print(f"Síntese factual: {len(compart)} lemas compartilhados com frequência combinada >= {FREQ_MINIMA_COMPARACAO}; "
      f"{int((razao < 1).sum())} ({num((razao < 1).mean() * 100, 1)}%) têm taxas que diferem menos que o dobro "
      "entre os documentos (entre as linhas claras).")

**Método e escolhas.** Cada ponto é um lema presente nos dois documentos, com frequência combinada mínima `FREQ_MINIMA_COMPARACAO`. A posição é dada pelas taxas por 1.000 tokens em cada documento, em escala logarítmica. A escala log permite ver no mesmo gráfico termos muito e pouco frequentes, e torna simétricas as razões (o dobro e a metade ficam à mesma distância da diagonal). São rotulados os `N_ROTULOS_DISPERSAO` lemas de maior taxa somada.

**Como ler.** A diagonal escura marca peso relativo **igual** nos dois documentos. Pontos sobre ela ou próximos dela são **semelhanças**; pontos fora das linhas claras têm taxa ao menos duas vezes maior em um dos documentos. Pontos mais à direita e mais acima são termos frequentes nos dois textos.

**Interpretação.** *[Pendente. A redigir somente após a execução do notebook com os JSONs definitivos da skill 01, exclusivamente a partir dos valores exibidos acima. Toda afirmação de diferença deve indicar o critério que a sustenta (skill 03, seção 4.3).]*

**Fragilidades e ajustes possíveis.** Lemas com frequências baixas formam padrões em "grade", porque as taxas assumem poucos valores possíveis. Pontos coincidentes ficam sobrepostos: a transparência indica a sobreposição, e não se usa deslocamento aleatório (*jitter*), que alteraria os valores. Termos exclusivos de um documento não aparecem aqui (ver seções 14 e 15). Os rótulos podem se sobrepor em regiões densas, e os valores exatos estão na tabela de *keyness* (`03_keyness_completa.csv`), exportada para `resultados/` apenas por comando do pesquisador.

## 14. Distinções: termos característicos de cada documento (*keyness*)

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
elegiveis = keyness[keyness["freq_combinada"] >= FREQ_MINIMA_COMPARACAO].copy()
elegiveis["distintivo"] = (elegiveis["G2"] >= LIMIAR_G2) & (elegiveis["log_ratio"].abs() >= LIMIAR_LOG_RATIO)
limiar_bonf = chi2.isf(ALFA_BONFERRONI / max(len(elegiveis), 1), 1)
elegiveis["distintivo_bonferroni"] = elegiveis["distintivo"] & (elegiveis["G2"] >= limiar_bonf)
salvar_tabela(elegiveis, "03_keyness_elegiveis")

dist = elegiveis[elegiveis["distintivo"]]
lado_a = dist[dist["log_ratio"] > 0].sort_values("G2", ascending=False).head(TOP_N_KEYNESS)
lado_b = dist[dist["log_ratio"] < 0].sort_values("G2", ascending=False).head(TOP_N_KEYNESS)
graf = pd.concat([lado_a.sort_values("log_ratio"), lado_b.sort_values("log_ratio")])

if graf.empty:
    print("Nenhum termo atende simultaneamente aos critérios de significância e tamanho de efeito.")
else:
    fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(graf) + 2.0), layout="constrained")
    y = np.arange(len(graf))
    cores = [COR[SIG_A] if v > 0 else COR[SIG_B] for v in graf["log_ratio"]]
    ax.barh(y, graf["log_ratio"], height=0.7, color=cores)
    rotulos = [f"{lema} (exclusivo)" if p != "ambos" else lema for lema, p in zip(graf.index, graf["presenca"])]
    ax.set_yticks(y, labels=rotulos)
    ax.axvline(0, color=COR_TINTA_2, lw=0.8)
    limite = graf["log_ratio"].abs().max() * 1.1          # eixo simétrico: os dois lados sempre visíveis
    ax.set_xlim(-limite, limite)
    ax.set_ylim(len(graf) - 0.5, -0.5)
    ax.xaxis.set_major_formatter(FMT_BR)
    ax.set_xlabel(f"Log Ratio (log₂ da razão entre taxas)\n←  mais em {SIG_B}  |  mais em {SIG_A}  →")
    ax.set_title(f"Termos distintivos (G² ≥ {num(LIMIAR_G2)} e |Log Ratio| ≥ {num(LIMIAR_LOG_RATIO, 1)})", loc="left")
    ax.grid(axis="y", visible=False)
    fig.legend(handles=[Line2D([0], [0], color=COR[s], lw=6, label=f"mais característico de {s}") for s in SIGLAS],
               loc="outside lower center", ncol=2)
    salvar_figura(fig, "03_keyness")
    plt.show()
    exibir(graf.iloc[::-1])

print(f"Síntese factual: {len(elegiveis)} lemas testados (frequência combinada >= {FREQ_MINIMA_COMPARACAO}); "
      f"{int((dist['log_ratio'] > 0).sum())} distintivos de {SIG_A} e {int((dist['log_ratio'] < 0).sum())} de {SIG_B}. "
      f"Após correção de Bonferroni (alfa = {num(ALFA_BONFERRONI)}; G² crítico = {num(limiar_bonf)}), "
      f"permanecem {int(elegiveis['distintivo_bonferroni'].sum())}.")

**Método e escolhas.** Para cada lema com frequência combinada mínima `FREQ_MINIMA_COMPARACAO`, calcula-se (a) o **G²** de log-verossimilhança (Dunning, 1993; Rayson & Garside, 2000), que testa se a diferença entre as taxas pode ser atribuída ao acaso, e (b) o **Log Ratio** (Hardie, 2014), que mede o tamanho da diferença (1 = o dobro; 2 = o quádruplo). Um termo é **distintivo** somente se passar pelos dois critérios: G² ≥ `LIMIAR_G2` (p < 0,01) **e** |Log Ratio| ≥ `LIMIAR_LOG_RATIO`. São exibidos até `TOP_N_KEYNESS` termos de cada lado, os de maior G². Para frequência zero, o Log Ratio usa correção de 0,5, e esses termos aparecem marcados como *exclusivo*. Também se informa quantos termos resistem à correção de Bonferroni para comparações múltiplas.

**Como ler.** Barras à direita (cor de BRICS) indicam termos proporcionalmente mais frequentes em BRICS; barras à esquerda (cor de OCDE), em OCDE. Quanto mais longa a barra, maior a diferença relativa. Esses termos indicam o que **distingue** cada texto.

**Interpretação.** *[Pendente. A redigir somente após a execução do notebook com os JSONs definitivos da skill 01, exclusivamente a partir dos valores exibidos acima. Toda afirmação de diferença deve indicar o critério que a sustenta (skill 03, seção 4.3).]*

**Fragilidades e ajustes possíveis.** O G² pressupõe independência entre ocorrências, o que não se verifica em textos (as palavras se repetem em blocos). Por isso, os valores de p são aproximados, e o critério combinado com tamanho de efeito é mais confiável que a significância isolada. Em documentos curtos, poucos termos passam pela correção de Bonferroni: o relato dessa contagem indica quão conservadora deve ser a interpretação. Termos distintivos podem refletir o **gênero textual** (por exemplo, vocabulário formular de recomendação ou de comunicado de cúpula), e não apenas diferença de posição. Os limiares são parâmetros editáveis e devem ser fixados antes da interpretação.

## 15. Vocabulário compartilhado e exclusivo

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
lemas = {s: set(freqs[s].index) for s in SIGLAS}
compartilhados = lemas[SIG_A] & lemas[SIG_B]
vocab = pd.DataFrame({
    "lemas de conteúdo distintos": {s: len(lemas[s]) for s in SIGLAS},
    "compartilhados": {s: len(compartilhados) for s in SIGLAS},
    "exclusivos": {s: len(lemas[s] - compartilhados) for s in SIGLAS},
    "exclusivos (% dos lemas do documento)": {s: len(lemas[s] - compartilhados) / len(lemas[s]) * 100 for s in SIGLAS},
    "tokens de conteúdo em lemas compartilhados (%)": {
        s: freqs[s].loc[sorted(compartilhados), "frequencia"].sum() / freqs[s]["frequencia"].sum() * 100 for s in SIGLAS},
}).T
exibir(vocab)
salvar_tabela(vocab, "04_vocabulario")

for s in SIGLAS:
    exclusivos = freqs[s].loc[sorted(lemas[s] - compartilhados)].sort_values(
        ["frequencia"], ascending=False, kind="mergesort").head(TOP_N_KEYNESS)
    print(f"\nLemas exclusivos de {s} mais frequentes:")
    exibir(exclusivos[["frequencia", "por_1000_tokens", "unidades_com_ocorrencia", "formas_observadas"]])

**Método e escolhas.** Contagem dos lemas de conteúdo presentes nos dois documentos (compartilhados) e em apenas um (exclusivos). A última linha da tabela mostra qual parcela das **ocorrências** de cada documento pertence a lemas compartilhados, uma medida menos sensível ao tamanho do que a contagem de lemas. Em seguida, listam-se os lemas exclusivos mais frequentes de cada documento.

**Como ler.** Uma parcela alta de ocorrências em lemas compartilhados indica um núcleo de vocabulário comum. Os lemas exclusivos mais frequentes indicam temas ou formulações presentes em apenas um dos textos.

**Interpretação.** *[Pendente. A redigir somente após a execução do notebook com os JSONs definitivos da skill 01, exclusivamente a partir dos valores exibidos acima. Toda afirmação de diferença deve indicar o critério que a sustenta (skill 03, seção 4.3).]*

**Fragilidades e ajustes possíveis.** **Ressalva de tamanho:** o número de lemas exclusivos cresce com a extensão do texto. O documento maior tende a ter mais lemas exclusivos simplesmente por ser maior, e por isso a contagem de exclusivos não deve ser lida como medida de diversidade temática. Lemas exclusivos com uma única ocorrência são frágeis como evidência.

## 16. Convergência e divergência de termos: unigramas

*Visualização construída por comando do pesquisador.*

In [ ]:
# ===========================================================================
# CONVERGÊNCIA E DIVERGÊNCIA DE TERMOS (seções 16 e 17), por comando do pesquisador.
# Usa os mesmos recortes dos itens 12 (lemas) e 14 (bigramas) dos notebooks individuais,
# em ocorrências por 1.000 tokens. Independente da seção 13.
# ===========================================================================
import textwrap

import matplotlib.patheffects as path_effects
from matplotlib.patches import Patch

# ---------------------------------------------------------------------------
# Parâmetros desta seção (editáveis pelo pesquisador)
# ---------------------------------------------------------------------------
TOP_N_LEMAS_CONVERGENCIA = 25      # recorte do item 12 dos notebooks individuais
TOP_N_BIGRAMAS_CONVERGENCIA = 20   # recorte do item 14 dos notebooks individuais
FREQ_MINIMA_BIGRAMA = 2            # frequência mínima do item 14 dos notebooks individuais
RAZAO_ZONA_NEUTRA = 2.0            # zona neutra: a taxa de um documento é menor que o dobro da do outro
FATOR_FORA_DA_ESCALA = 2.0         # termo com taxa maior que o dobro da maior taxa dos demais fica fora da escala
N_DESTAQUES_NEUTROS = 4            # termos da zona neutra rotulados além dos pedidos (maior taxa somada)
COR_NEUTRA = "#8c8a85"             # cinza da zona neutra (laranja e azul são as cores fixas dos documentos)
COR_FAIXA = "#ecebe6"              # preenchimento da faixa da zona neutra
NOME_NO = {"BRICS": "no BRICS", "OCDE": "na OCDE"}   # forma usada nos textos do gráfico

# Termos destacados por pedido do pesquisador, procurados pelo lema ou por uma forma escrita.
# A cor de cada termo segue os dados (zona), e não o grupo em que foi pedido.
DESTAQUES_UNIGRAMAS = {
    "OCDE": ["recommendation", "policy", "stakeholder", "trustworthy", "principle", "responsible", "human"],
    "BRICS": ["development", "governance", "inclusive", "promote", "global", "sustainable", "framework"],
}
DESTAQUES_BIGRAMAS = {
    "OCDE": ["trustworthy ai", "system lifecycle", "human rights", "policy environment",
             "responsible stewardship", "data protection", "foster multistakeholder"],
    "BRICS": ["developing countries", "ai governance", "digital economy", "sustainable development",
              "global ai",   # no pedido: "global IA" (IA = AI)
              "applicable international", "data protection"],
}


# ---------------------------------------------------------------------------
# Dados do gráfico
# ---------------------------------------------------------------------------
def normalizar_termo(texto):
    """Minúsculas, sem hífen e com espaços simples (ex.: 'Multi-Stakeholder' -> 'multistakeholder')."""
    return " ".join(str(texto).lower().replace("-", "").split())


def contar_formas(*textos):
    """Soma as contagens de formas escritas no formato 'forma (n); forma (n)' de um ou mais documentos."""
    contagem = {}
    for texto in textos:
        if not isinstance(texto, str) or not texto:
            continue
        for item in texto.split("; "):
            forma, n = re.fullmatch(r"(.*) \((\d+)\)", item).groups()
            contagem[forma] = contagem.get(forma, 0) + int(n)
    return contagem


def zona_do_termo(freq_a, freq_b, taxa_a, taxa_b):
    """Zona do termo pela razão entre as taxas: SIG_A ou SIG_B se a taxa de um documento for ao menos
    RAZAO_ZONA_NEUTRA vezes a do outro (ou se o termo só ocorrer nele); 'neutra' nos demais casos."""
    if freq_a > 0 and freq_b == 0:
        return SIG_A
    if freq_b > 0 and freq_a == 0:
        return SIG_B
    if taxa_a >= RAZAO_ZONA_NEUTRA * taxa_b:
        return SIG_A
    if taxa_b >= RAZAO_ZONA_NEUTRA * taxa_a:
        return SIG_B
    return "neutra"


def preparar_convergencia(tab_a, tab_b, recortes, pedidos, rotulo_por_forma):
    """Une os recortes 'top N' dos dois documentos e calcula, para cada termo: ocorrências, taxas por
    1.000 tokens, G² e Log Ratio (mesma função da seção 14), zona, destaque e posição na escala.
    Termos pedidos pelo pesquisador que estejam fora dos recortes são acrescentados e sinalizados."""
    termos = list(dict.fromkeys(list(recortes[SIG_A]) + list(recortes[SIG_B])))
    origem = {}
    for t in termos:
        dentro = [s for s in SIGLAS if t in set(recortes[s])]
        origem[t] = "recortes " + " e ".join(dentro) if len(dentro) == 2 else f"recorte {dentro[0]}"

    # Termos pedidos: procurados pelo termo (lema) ou por qualquer forma escrita observada.
    todos = tab_a.index.union(tab_b.index)
    formas = {t: contar_formas(tab_a["formas_observadas"].get(t), tab_b["formas_observadas"].get(t)) for t in todos}
    chaves = {t: {normalizar_termo(t)} | {normalizar_termo(f) for f in formas[t]} for t in todos}
    pedido_por, nao_encontrados = {}, []
    for grupo, lista in pedidos.items():
        for pedido in lista:
            alvo = normalizar_termo(pedido)
            achados = [t for t in todos if alvo in chaves[t]]
            if not achados:
                nao_encontrados.append(f"{pedido} ({grupo})")
                continue
            plotados = [t for t in achados if t in termos]
            if plotados:
                escolhido = plotados[0]
            else:   # fora dos recortes: entra no gráfico por pedido do pesquisador
                escolhido = max(achados, key=lambda t: (tab_a["frequencia"].get(t, 0) + tab_b["frequencia"].get(t, 0), t))
                termos.append(escolhido)
                origem[escolhido] = "pedido do pesquisador"
            pedido_por.setdefault(escolhido, []).append(grupo)

    tab = tabela_keyness(tab_a, tab_b, N[SIG_A], N[SIG_B], SIG_A, SIG_B).reindex(termos)
    tab.index.name = "termo"
    tab["x"] = tab[f"por_1000_{SIG_A}"]
    tab["y"] = tab[f"por_1000_{SIG_B}"]
    tab["zona"] = [zona_do_termo(fa, fb, xa, yb) for fa, fb, xa, yb in
                   zip(tab[f"freq_{SIG_A}"], tab[f"freq_{SIG_B}"], tab["x"], tab["y"])]
    if rotulo_por_forma:   # bigramas: forma escrita mais frequente nos dois documentos (desempate alfabético)
        tab["rotulo"] = [sorted(formas[t].items(), key=lambda kv: (-kv[1], kv[0]))[0][0] for t in tab.index]
    else:                  # lemas: o próprio lema, como no item 12
        tab["rotulo"] = tab.index
    tab["origem"] = [origem[t] for t in tab.index]
    tab["pedido"] = [" e ".join(dict.fromkeys(pedido_por.get(t, []))) for t in tab.index]

    # Fora da escala: termo cuja maior taxa supera FATOR_FORA_DA_ESCALA vezes a maior taxa dos demais.
    maiores = tab[["x", "y"]].max(axis=1).sort_values(ascending=False)
    fora = []
    while len(maiores) > 1 and maiores.iloc[0] > FATOR_FORA_DA_ESCALA * maiores.iloc[1]:
        fora.append(maiores.index[0])
        maiores = maiores.iloc[1:]
    tab["fora_da_escala"] = tab.index.isin(fora)

    # Destaques: os pedidos e, na zona neutra, até N_DESTAQUES_NEUTROS de maior taxa somada (desempate alfabético).
    neutros = tab[(tab["zona"] == "neutra") & (tab["pedido"] == "") & ~tab["fora_da_escala"]].copy()
    neutros["soma"] = neutros["x"] + neutros["y"]
    neutros = neutros.reset_index().sort_values(["soma", "termo"], ascending=[False, True]).head(N_DESTAQUES_NEUTROS)
    tab["destaque"] = np.where(tab["pedido"] != "", "pedido do pesquisador",
                               np.where(tab.index.isin(neutros["termo"]), "zona neutra (regra)", ""))
    tab["distintivo_secao_14"] = ((tab["G2"] >= LIMIAR_G2) & (tab["log_ratio"].abs() >= LIMIAR_LOG_RATIO)
                                  & (tab["freq_combinada"] >= FREQ_MINIMA_COMPARACAO))
    tab.attrs["nao_encontrados"] = nao_encontrados
    return tab


# ---------------------------------------------------------------------------
# Posicionamento de rótulos sem sobreposição
# ---------------------------------------------------------------------------
def _sobrepoe(a, b):
    """As caixas (x0, y0, x1, y1) se sobrepõem?"""
    return a[0] < b[2] and b[0] < a[2] and a[1] < b[3] and b[1] < a[3]


def _cruza(p, q, caixa):
    """O segmento p-q atravessa a caixa? (recorte de Liang-Barsky)"""
    (x0, y0), (x1, y1) = p, q
    dx, dy = x1 - x0, y1 - y0
    t0, t1 = 0.0, 1.0
    for pk, qk in ((-dx, x0 - caixa[0]), (dx, caixa[2] - x0), (-dy, y0 - caixa[1]), (dy, caixa[3] - y0)):
        if pk == 0:
            if qk < 0:
                return False
            continue
        t = qk / pk
        if pk < 0:
            t0 = max(t0, t)
        else:
            t1 = min(t1, t)
        if t0 > t1:
            return False
    return True


def _distancia_caixa(p, caixa):
    """Distância de um ponto a uma caixa (0 se estiver dentro)."""
    dx = max(caixa[0] - p[0], 0, p[0] - caixa[2])
    dy = max(caixa[1] - p[1], 0, p[1] - caixa[3])
    return float(np.hypot(dx, dy))


def _segmentos_cruzam(p1, p2, q1, q2):
    """Os segmentos p1-p2 e q1-q2 se cruzam? (pontas comuns não contam)"""
    def orient(a, b, c):
        return (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])
    return (orient(q1, q2, p1) * orient(q1, q2, p2) < 0) and (orient(p1, p2, q1) * orient(p1, p2, q2) < 0)


class Rotulador:
    """Posiciona rótulos sem sobreposição. Cada rótulo tem posições candidatas ao redor do seu ponto, da mais
    próxima para a mais distante, e cada posição tem um custo: distância ao ponto e sobreposição a pontos,
    a outros rótulos e a linhas de ligação (proibitivas), além de passagem sobre a diagonal ou os limites da
    faixa (evitada). Um rótulo sem linha de ligação não pode ficar mais perto de outro ponto do que do seu.
    Os rótulos são distribuídos em várias ordens (a primeira por densidade; as demais sorteadas com semente
    fixa) e fica a distribuição de menor custo total. O resultado é sempre o mesmo para os mesmos dados."""

    ANGULOS = (0, 35, -35, 70, -70, 110, -110, 145, -145, 180, 90, -90)
    DISTANCIAS_PT = (3, 6, 10, 15, 21, 28, 37, 48, 61, 76, 94, 115, 140)

    def __init__(self, ax, pontos_dados, raios_pt, linhas_suaves_dados):
        self.ax = ax
        self.renderer = ax.figure.canvas.get_renderer()
        self.px = ax.figure.dpi / 72
        self.pontos = ax.transData.transform(pontos_dados)
        self.raios = np.asarray(raios_pt, dtype=float) * self.px
        self.caixas_pontos = [(x - r - 1, y - r - 1, x + r + 1, y + r + 1) for (x, y), r in zip(self.pontos, self.raios)]
        e = ax.get_window_extent(self.renderer)
        self.limite = (e.x0 + 2, e.y0 + 2, e.x1 - 2, e.y1 - 2)
        self.linhas_suaves = [tuple(map(tuple, ax.transData.transform(seg))) for seg in linhas_suaves_dados]
        self.ocupadas, self.segmentos = [], []

    def caixa(self, artista, folga=2):
        bb = artista.get_window_extent(self.renderer)
        return (bb.x0 - folga, bb.y0 - folga, bb.x1 + folga, bb.y1 + folga)

    def ocupar(self, artista):
        self.ocupadas.append(self.caixa(artista))

    def ocupar_segmento(self, p_dados, q_dados):
        p, q = self.ax.transData.transform([p_dados, q_dados])
        self.segmentos.append((tuple(p), tuple(q)))

    def _candidatos(self, alvo, raio, largura, altura, proprio, distancias):
        """Posições candidatas de um rótulo, com o custo que não depende dos outros rótulos a posicionar."""
        candidatos = []
        for d_pt in distancias:
            for k, ang in enumerate(self.ANGULOS):
                u = np.array([np.cos(np.radians(ang)), np.sin(np.radians(ang))])
                ancora = alvo + (raio + d_pt * self.px) * u
                ha = "left" if u[0] > 0.3 else "right" if u[0] < -0.3 else "center"
                va = "bottom" if u[1] > 0.3 else "top" if u[1] < -0.3 else "center"
                x0 = ancora[0] - {"left": 0, "right": largura, "center": largura / 2}[ha]
                y0 = ancora[1] - {"bottom": 0, "top": altura, "center": altura / 2}[va]
                caixa = (x0 - 2, y0 - 2, x0 + largura + 2, y0 + altura + 2)
                ligacao = None
                if d_pt >= 6 and proprio >= 0:
                    ligacao = (tuple(alvo), (float(np.clip(alvo[0], caixa[0] + 2, caixa[2] - 2)),
                                             float(np.clip(alvo[1], caixa[1] + 2, caixa[3] - 2))))
                custo = d_pt + 2.0 * k
                if not (caixa[0] >= self.limite[0] and caixa[1] >= self.limite[1]
                        and caixa[2] <= self.limite[2] and caixa[3] <= self.limite[3]):
                    custo += 5000
                custo += 1000 * sum(_sobrepoe(caixa, c) for c in self.ocupadas)
                custo += 400 * sum(_sobrepoe(caixa, c) for c in self.caixas_pontos)
                custo += 600 * sum(_cruza(p, q, caixa) for p, q in self.segmentos)
                custo += 25 * sum(_cruza(p, q, caixa) for p, q in self.linhas_suaves)
                outros = [_distancia_caixa(p, caixa) - r for i, (p, r) in enumerate(zip(self.pontos, self.raios)) if i != proprio]
                if ligacao is None:
                    # Sem ligação, o rótulo precisa estar claramente mais perto do seu ponto.
                    if proprio >= 0 and outros and min(outros) < _distancia_caixa(alvo, caixa) + 4:
                        custo += 60
                else:
                    # Com ligação, o rótulo também não deve encostar em outro ponto (leitura ambígua).
                    if outros and min(outros) < 7:
                        custo += 80
                    custo += 600 * sum(_cruza(ligacao[0], ligacao[1], c) for c in self.ocupadas)
                    custo += 300 * sum(_cruza(ligacao[0], ligacao[1], c) for i, c in enumerate(self.caixas_pontos) if i != proprio)
                candidatos.append((custo, ancora, ha, va, caixa, ligacao))
        candidatos.sort(key=lambda c: c[0])
        return candidatos[:40]

    @staticmethod
    def _custo_mutuo(candidato, colocados):
        """Custo de um candidato diante dos rótulos já colocados nesta distribuição."""
        caixa, ligacao = candidato[4], candidato[5]
        custo = 0.0
        for caixa_2, ligacao_2 in colocados:
            if _sobrepoe(caixa, caixa_2):
                custo += 1000
            if ligacao_2 is not None and _cruza(ligacao_2[0], ligacao_2[1], caixa):
                custo += 600
            if ligacao is not None:
                if _cruza(ligacao[0], ligacao[1], caixa_2):
                    custo += 600
                if ligacao_2 is not None and _segmentos_cruzam(ligacao[0], ligacao[1], ligacao_2[0], ligacao_2[1]):
                    custo += 200
        return custo

    def posicionar(self, pedidos, tentativas=150, semente=0, distancias=None):
        """Cria os rótulos de 'pedidos' — lista de (xy, texto, cor_da_ligacao, indice_do_ponto, estilo);
        índice -1 indica uma âncora que não é ponto (ex.: a seta de um termo fora da escala)."""
        if not pedidos:
            return []
        artistas, candidatos = [], []
        for xy, texto, _, proprio, estilo in pedidos:
            artista = self.ax.text(0, 0, texto, zorder=6, clip_on=False, ha="left", va="bottom", **estilo)
            bb = artista.get_window_extent(self.renderer)
            alvo = self.ax.transData.transform([xy])[0]
            raio = self.raios[proprio] if proprio >= 0 else 0.0
            candidatos.append(self._candidatos(alvo, raio, bb.width, bb.height, proprio, distancias or self.DISTANCIAS_PT))
            artistas.append(artista)

        alvos = self.ax.transData.transform([p[0] for p in pedidos])
        densidade = [int((np.hypot(*(self.pontos - a).T) < 45 * self.px).sum()) for a in alvos]
        ordens = [sorted(range(len(pedidos)), key=lambda i: -densidade[i])]
        rng = np.random.default_rng(semente)
        ordens += [list(rng.permutation(len(pedidos))) for _ in range(tentativas)]
        melhor = None
        for ordem in ordens:
            escolha, colocados, total = {}, [], 0.0
            for i in ordem:
                custo, candidato = min(((c[0] + self._custo_mutuo(c, colocados), c) for c in candidatos[i]),
                                       key=lambda par: par[0])
                escolha[i] = candidato
                colocados.append((candidato[4], candidato[5]))
                total += custo
                if melhor is not None and total >= melhor[0]:
                    break
            else:
                if melhor is None or total < melhor[0]:
                    melhor = (total, escolha)

        inv = self.ax.transData.inverted()
        for i, (artista, (xy, _, cor, proprio, _)) in enumerate(zip(artistas, pedidos)):
            _, ancora, ha, va, caixa, ligacao = melhor[1][i]
            artista.set_position(inv.transform(ancora))
            artista.set_ha(ha)
            artista.set_va(va)
            self.ocupadas.append(caixa)
            if ligacao is not None:
                self.segmentos.append(ligacao)
                raio_pt = self.raios[proprio] / self.px if proprio >= 0 else 0
                self.ax.annotate("", xy=xy, xytext=inv.transform(ligacao[1]), zorder=5, annotation_clip=False,
                                 arrowprops=dict(arrowstyle="-", color=cor, lw=0.7, shrinkA=0, shrinkB=raio_pt + 1))
        return artistas


# ---------------------------------------------------------------------------
# Gráfico
# ---------------------------------------------------------------------------
def grafico_convergencia(tab, subtitulo, descricao_ponto):
    """Gráfico de convergência e divergência: X = taxa em SIG_A, Y = taxa em SIG_B (por 1.000 tokens),
    mesma escala nos dois eixos, diagonal de frequência igual e faixa da zona neutra."""
    cor_zona = {SIG_A: COR[SIG_A], SIG_B: COR[SIG_B], "neutra": COR_NEUTRA}
    r_txt = f"{RAZAO_ZONA_NEUTRA:g}".replace(".", ",")
    dentro = tab[~tab["fora_da_escala"]]
    lim = MaxNLocator(nbins=7, steps=[1, 2, 2.5, 5, 10]).tick_values(0, dentro[["x", "y"]].to_numpy().max() * 1.1)[-1]

    largura, altura, lado, esq, base = 7.6, 9.4, 6.0, 1.0, 2.75   # polegadas
    fig = plt.figure(figsize=(largura, altura))
    ax = fig.add_axes([esq / largura, base / altura, lado / largura, lado / altura])
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_aspect("equal", adjustable="box")

    # Zona neutra (faixa entre y = x / R e y = R·x) e diagonal de frequência igual.
    xs = np.linspace(0, lim, 400)
    ax.fill_between(xs, xs / RAZAO_ZONA_NEUTRA, np.minimum(xs * RAZAO_ZONA_NEUTRA, lim),
                    color=COR_FAIXA, alpha=0.75, lw=0, zorder=0)
    for fator in (RAZAO_ZONA_NEUTRA, 1 / RAZAO_ZONA_NEUTRA):
        ax.plot(xs, xs * fator, color=COR_LIGACAO, lw=0.7, ls=(0, (1.5, 2)), zorder=1)
    ax.plot([0, lim], [0, lim], color=COR_TINTA_2, lw=1.0, ls=(0, (5, 4)), zorder=2)

    # Pontos. Termos com os mesmos valores formam um único ponto, maior, com o número de termos dentro.
    grupos = []
    for (x, y), g in dentro.groupby([dentro["x"].round(9), dentro["y"].round(9)], sort=False):
        grupos.append({"x": g["x"].iloc[0], "y": g["y"].iloc[0], "zona": g["zona"].iloc[0], "n": len(g),
                       "destacados": sorted(g.loc[g["destaque"] != "", "rotulo"])})
    tam_simples, tam_multiplo = 46, 150
    for zona in ("neutra", SIG_A, SIG_B):
        for multiplo in (False, True):
            sel = [p for p in grupos if p["zona"] == zona and (p["n"] > 1) == multiplo]
            if sel:
                ax.scatter([p["x"] for p in sel], [p["y"] for p in sel], s=tam_multiplo if multiplo else tam_simples,
                           color=cor_zona[zona], alpha=0.92, edgecolors="white", linewidths=0.8, zorder=3, clip_on=False)
    for p in grupos:
        if p["n"] > 1:
            ax.text(p["x"], p["y"], str(p["n"]), ha="center", va="center", fontsize=7.5, fontweight="bold",
                    color="white", zorder=4, clip_on=False)

    ax.xaxis.set_major_locator(MaxNLocator(nbins=7, steps=[1, 2, 2.5, 5, 10]))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=7, steps=[1, 2, 2.5, 5, 10]))
    ax.xaxis.set_major_formatter(FMT_BR)
    ax.yaxis.set_major_formatter(FMT_BR)
    ax.set_xlabel(f"Frequência relativa {NOME_NO[SIG_A]} (ocorrências por 1.000 tokens)", labelpad=8)
    ax.set_ylabel(f"Frequência relativa {NOME_NO[SIG_B]} (ocorrências por 1.000 tokens)", labelpad=8)
    centro = (esq + lado / 2) / largura   # título centrado sobre a área do gráfico
    fig.text(centro, 1 - 0.18 / altura, f"Convergência e divergência de termos — {SIG_A} × {SIG_B}",
             ha="center", va="top", fontsize=13, fontweight="bold")
    fig.text(centro, 1 - 0.48 / altura, subtitulo, ha="center", va="top", fontsize=10, color=COR_TINTA_2)
    fig.canvas.draw()

    contorno = [path_effects.withStroke(linewidth=3, foreground="white")]
    raios = [np.sqrt(tam_multiplo if p["n"] > 1 else tam_simples) / 2 for p in grupos]
    rot = Rotulador(ax, np.array([(p["x"], p["y"]) for p in grupos]), raios,
                    [[(0, 0), (lim, lim)], [(0, 0), (lim / RAZAO_ZONA_NEUTRA, lim)], [(0, 0), (lim, lim / RAZAO_ZONA_NEUTRA)]])

    # Nome das regiões: no primeiro canto livre de pontos, inteiramente dentro da zona de cada documento.
    inv = ax.transData.inverted()
    for zona, texto, candidatos in (
        (SIG_B, f"↑ mais frequente {NOME_NO[SIG_B]}",
         [(0.03, f, "left", "top") for f in (0.97, 0.91, 0.85, 0.79, 0.73, 0.67)]),
        (SIG_A, f"mais frequente {NOME_NO[SIG_A]} →",
         [(0.97, f, "right", "bottom") for f in (0.03, 0.09, 0.15, 0.21, 0.27, 0.33)]),
    ):
        artista = ax.text(0, 0, texto, transform=ax.transAxes, fontsize=9.5, style="italic",
                          color=cor_zona[zona], zorder=6, path_effects=contorno)
        escolhido = None
        for fx, fy, ha, va in candidatos:
            artista.set_position((fx, fy))
            artista.set_ha(ha)
            artista.set_va(va)
            caixa = rot.caixa(artista, folga=6)   # folga maior: o nome da região não encosta em pontos
            (x0, y0), (x1, y1) = inv.transform([(caixa[0], caixa[1]), (caixa[2], caixa[3])])
            na_zona = y0 >= RAZAO_ZONA_NEUTRA * x1 if zona == SIG_B else y1 <= x0 / RAZAO_ZONA_NEUTRA
            livre = not any(_sobrepoe(caixa, c) for c in rot.caixas_pontos)
            if na_zona and livre:
                escolhido = (fx, fy, ha, va)
                break
        fx, fy, ha, va = escolhido or candidatos[0]
        artista.set_position((fx, fy))
        artista.set_ha(ha)
        artista.set_va(va)
        rot.ocupar(artista)

    # Termos fora da escala: seta na borda, na direção do valor real, com os valores exatos.
    for termo, linha in tab[tab["fora_da_escala"]].iterrows():
        ponta = np.array([min(linha["x"], lim), min(linha["y"], lim)])
        direcao = np.array([linha["x"] > lim, linha["y"] > lim], dtype=float)
        direcao /= np.linalg.norm(direcao)
        cauda = ponta - direcao * lim * 0.075
        ax.annotate("", xy=ponta, xytext=cauda, zorder=6, annotation_clip=False,
                    arrowprops=dict(arrowstyle="-|>", color=cor_zona[linha["zona"]], lw=1.6, mutation_scale=12))
        rot.ocupar_segmento(cauda, ponta)
        rot.posicionar([(cauda, f"{linha['rotulo']} (fora da escala)\n{SIG_A}: {num(linha['x'])} · {SIG_B}: {num(linha['y'])}",
                         cor_zona[linha["zona"]], -1,
                         dict(fontsize=9, color=COR_TINTA, linespacing=1.3, path_effects=contorno))],
                       tentativas=0, distancias=(2, 4, 7, 11))

    # Rótulos dos termos destacados: um por ponto (termos com os mesmos valores dividem o rótulo).
    estilo = dict(fontsize=9, color=COR_TINTA, linespacing=1.15, path_effects=contorno)
    rot.posicionar([((p["x"], p["y"]), "\n".join(p["destacados"]), cor_zona[p["zona"]], i, estilo)
                    for i, p in enumerate(grupos) if p["destacados"]], tentativas=150, semente=SEMENTE)

    # Legenda e instrução de leitura, abaixo do gráfico.
    marcador = dict(ls="", marker="o", ms=7.5, mec="white", mew=0.8)
    itens = [
        Line2D([], [], mfc=COR[SIG_A], label=f"Mais frequente {NOME_NO[SIG_A]}: taxa {NOME_NO[SIG_A]} ≥ {r_txt}× a taxa {NOME_NO[SIG_B]}", **marcador),
        Line2D([], [], mfc=COR[SIG_B], label=f"Mais frequente {NOME_NO[SIG_B]}: taxa {NOME_NO[SIG_B]} ≥ {r_txt}× a taxa {NOME_NO[SIG_A]}", **marcador),
        Line2D([], [], mfc=COR_NEUTRA, label=f"Zona neutra: usado por ambos, com diferença menor que {r_txt}×", **marcador),
        Line2D([], [], color=COR_TINTA_2, lw=1.0, ls=(0, (5, 4)), label="Linha tracejada: frequência igual nos dois documentos (x = y)"),
        Patch(facecolor=COR_FAIXA, edgecolor=COR_LIGACAO, lw=0.7, ls=":", label=f"Faixa cinza: zona neutra, entre y = x/{r_txt} e y = {r_txt}·x"),
    ]
    fig.legend(handles=itens, loc="upper left", bbox_to_anchor=(esq / largura, (base - 0.62) / altura),
               ncol=1, fontsize=9, handlelength=2.2, handletextpad=0.8, labelspacing=0.45, borderaxespad=0)
    leitura = (f"Como ler: cada ponto é um {descricao_ponto}. O eixo X mostra quantas vezes o termo aparece "
               f"a cada 1.000 tokens (palavras) {NOME_NO[SIG_A]}; o eixo Y, o mesmo {NOME_NO[SIG_B]}. "
               f"Ex.: um ponto em x = 4 e y = 2 indica 4 ocorrências a cada 1.000 palavras {NOME_NO[SIG_A]} "
               f"e 2 {NOME_NO[SIG_B]}. Acima da diagonal, o termo é proporcionalmente mais frequente {NOME_NO[SIG_B]}; "
               f"abaixo, {NOME_NO[SIG_A]}. Pontos sobre um eixo (x = 0 ou y = 0) aparecem em um só documento. "
               "Um ponto maior com um número dentro reúne esse número de termos com exatamente os mesmos valores.")
    fig.text(esq / largura, (base - 1.75) / altura, textwrap.fill(leitura, 100), ha="left", va="top",
             fontsize=9, color=COR_TINTA_2, linespacing=1.45)
    return fig


def tabela_convergencia_exibicao(tab, nome_indice):
    """Tabela com os valores exatos do gráfico (uma linha por termo), ordenada por zona e taxa."""
    ordem_zona = {SIG_A: 0, "neutra": 1, SIG_B: 2}
    t = tab.copy()
    t["_ordem"] = t["zona"].map(ordem_zona)
    t["_taxa"] = np.where(t["zona"] == SIG_B, t["y"], np.where(t["zona"] == SIG_A, t["x"], t["x"] + t["y"]))
    t = t.sort_values(["_ordem", "_taxa"], ascending=[True, False])
    saida = pd.DataFrame({
        "rótulo": t["rotulo"],
        f"{SIG_A}: ocorrências": t[f"freq_{SIG_A}"],
        f"{SIG_A}: por 1.000 tokens": t["x"],
        f"{SIG_B}: ocorrências": t[f"freq_{SIG_B}"],
        f"{SIG_B}: por 1.000 tokens": t["y"],
        "Log Ratio": t["log_ratio"],
        "G²": t["G2"],
        "zona": t["zona"],
        "distintivo (critérios da seção 14)": np.where(t["distintivo_secao_14"], "sim", "não"),
        "origem": t["origem"],
        "pedido como destaque de": t["pedido"].replace("", "—"),
        "rotulado": np.where(t["destaque"] != "", t["destaque"], "—"),
        "fora da escala": np.where(t["fora_da_escala"], "sim", "—"),
    })
    saida.index.name = nome_indice
    return saida


def sintese_convergencia(tab, nome_plural, tops):
    """Síntese factual do gráfico: contagens por zona, termos fora da escala e conferência dos destaques."""
    contagem = tab["zona"].value_counts()
    ambos = int((tab["origem"] == f"recortes {SIG_A} e {SIG_B}").sum())
    print(f"Síntese factual: {len(tab)} {nome_plural} no gráfico (união dos {tops} mais frequentes de cada documento; "
          f"{ambos} estão nos dois recortes). Zona {SIG_A}: {contagem.get(SIG_A, 0)}; zona neutra: "
          f"{contagem.get('neutra', 0)}; zona {SIG_B}: {contagem.get(SIG_B, 0)}.")
    for termo, l in tab[tab["fora_da_escala"]].iterrows():
        print(f"Fora da escala (indicado por seta): {l['rotulo']} — {SIG_A} {num(l['x'])}; {SIG_B} {num(l['y'])} "
              f"por 1.000 tokens; zona {l['zona']}.")
    acrescentados = tab[tab["origem"] == "pedido do pesquisador"]
    if len(acrescentados):
        print("Acrescentados por pedido do pesquisador (fora dos recortes): " + ", ".join(acrescentados["rotulo"]) + ".")
    for item in tab.attrs.get("nao_encontrados", []):
        print(f"Pedido não encontrado em nenhum dos documentos: {item}.")
    divergentes = tab[(tab["pedido"] != "") & (tab["pedido"].str.contains(" e ") | (tab["pedido"] != tab["zona"]))]
    for termo, l in divergentes.iterrows():
        razao = max(l["x"], l["y"]) / min(l["x"], l["y"]) if min(l["x"], l["y"]) > 0 else float("inf")
        print(f"Atenção: '{l['rotulo']}' foi pedido como destaque de {l['pedido']}; pelos dados, está na zona "
              f"{l['zona']} ({SIG_A} {num(l['x'])}; {SIG_B} {num(l['y'])}; razão {num(razao)}×).")
    neutros = tab[tab["destaque"] == "zona neutra (regra)"]
    print(f"Rotulados na zona neutra pela regra (até {N_DESTAQUES_NEUTROS}, de maior taxa somada): "
          + (", ".join(neutros["rotulo"]) or "(nenhum)") + ".")
    print(f"Passam nos critérios de keyness da seção 14 (G² ≥ {num(LIMIAR_G2)}, |Log Ratio| ≥ {num(LIMIAR_LOG_RATIO, 1)} "
          f"e frequência combinada ≥ {FREQ_MINIMA_COMPARACAO}): {int(tab['distintivo_secao_14'].sum())} de {len(tab)}.")

In [ ]:
# Unigramas: os 25 lemas de conteúdo mais frequentes de cada documento (mesmo recorte do item 12
# dos notebooks individuais), em ocorrências por 1.000 tokens.
for s in SIGLAS:
    relatar_empates(freqs[s], "frequencia", TOP_N_LEMAS_CONVERGENCIA, f"lema de {s}")
recortes_lemas = {s: freqs[s].head(TOP_N_LEMAS_CONVERGENCIA).index for s in SIGLAS}
conv_lemas = preparar_convergencia(freqs[SIG_A], freqs[SIG_B], recortes_lemas, DESTAQUES_UNIGRAMAS,
                                   rotulo_por_forma=False)

fig = grafico_convergencia(
    conv_lemas,
    subtitulo=(f"Unigramas: os {TOP_N_LEMAS_CONVERGENCIA} lemas de conteúdo mais frequentes de cada documento "
               f"(união: {len(conv_lemas)} lemas)"),
    descricao_ponto="lema de conteúdo (unigrama)",
)
salvar_figura(fig, "05_convergencia_unigramas")
plt.show()

tabela_lemas = tabela_convergencia_exibicao(conv_lemas, "lema")
salvar_tabela(tabela_lemas, "05_convergencia_unigramas")
exibir(tabela_lemas)
sintese_convergencia(conv_lemas, "lemas", TOP_N_LEMAS_CONVERGENCIA)

**Método e escolhas.** Os termos são os mesmos do item 12 dos notebooks individuais: os 25 lemas de conteúdo mais frequentes de cada documento (`TOP_N_LEMAS_CONVERGENCIA`), com o mesmo pré-processamento e o mesmo desempate (ordem alfabética, informada acima do gráfico quando afeta o corte), reunidos em uma única lista (união dos dois recortes). Cada lema é posicionado pela sua **frequência relativa**: X = ocorrências por 1.000 tokens no BRICS; Y = ocorrências por 1.000 tokens na OCDE. O denominador é o total de tokens de cada documento, incluindo *stopwords*. Os dois eixos têm a mesma escala e começam em zero, e a diagonal tracejada marca frequência igual nos dois documentos. A cor indica a zona do termo, por regra fixa (`RAZAO_ZONA_NEUTRA` = 2, equivalente ao limiar |*Log Ratio*| ≥ 1 da seção 14): **laranja** quando a taxa no BRICS é ao menos o dobro da taxa na OCDE; **azul** no caso inverso; **cinza** (zona neutra) quando a diferença é menor que o dobro. A faixa cinza mostra os limites dessa zona, e termos ausentes de um documento ficam sobre o eixo do outro. São rotulados os termos pedidos pelo pesquisador (`DESTAQUES_UNIGRAMAS`) e, na zona neutra, até `N_DESTAQUES_NEUTROS` outros termos, os de maior taxa somada. A cor de cada termo segue os dados, e não o grupo em que foi pedido: a síntese acima informa os pedidos que, pelos dados, ficaram em outra zona. Um termo cuja taxa supera em mais que o dobro a maior taxa dos demais (`FATOR_FORA_DA_ESCALA`) fica fora da escala e é indicado por seta, com os valores exatos, para não comprimir os demais pontos. Termos com exatamente os mesmos valores formam um único ponto, maior, com o número de termos dentro. A posição dos rótulos é calculada automaticamente para não cobrir pontos nem outros rótulos, com semente fixa (`SEMENTE`); rótulos afastados do ponto recebem uma linha de ligação. A tabela traz os valores exatos de cada termo, o *Log Ratio* e o G² (mesma função da seção 14) e indica se o termo passa nos critérios de *keyness* daquela seção. Este gráfico não usa a seção 13, que trata de todos os lemas compartilhados, em escala logarítmica.

**Como ler.** Cada ponto é um lema. Quanto mais à direita, mais frequente no BRICS; quanto mais acima, mais frequente na OCDE. Pontos próximos da diagonal têm peso relativo parecido nos dois textos (convergência); pontos afastados dela, em direção a um dos eixos, indicam ênfase de um dos documentos (divergência).

**Interpretação.** *[Pendente. A redigir somente após a execução do notebook com os JSONs definitivos da skill 01, exclusivamente a partir dos valores exibidos acima. Toda afirmação de diferença deve indicar o critério que a sustenta (skill 03, seção 4.3).]*

**Fragilidades e ajustes possíveis.** A zona é **descritiva** (razão entre taxas) e **não** é um teste de significância: um termo pode ficar fora da zona neutra sem passar pelos critérios de *keyness* da seção 14 (G² e *Log Ratio*), e a tabela informa quais passam. O limiar do dobro é uma convenção: termos próximos das linhas de fronteira mudam de zona se o limiar for alterado. Só aparecem os termos da união dos recortes; termos relevantes fora dos 25 mais frequentes de cada documento não entram, e o resultado depende de *N*. Lemas genéricos, como *ai*, tendem a ocupar o topo nas declarações sobre IA; não são removidos sem decisão do pesquisador e, quando superam muito os demais, ficam fora da escala. Frequência relativa não equivale a importância política, e as diferenças podem refletir o gênero textual, a extensão e a função de cada documento, e não apenas posições substantivas.

## 17. Convergência e divergência de termos: bigramas

*Visualização construída por comando do pesquisador.*

In [ ]:
# Bigramas: as 20 expressões de duas palavras mais frequentes de cada documento, com frequência
# mínima 2 (mesmo recorte do item 14 dos notebooks individuais), em ocorrências por 1.000 tokens.
bigramas_doc = {s: tabela_bigramas(tokens[s]) for s in SIGLAS}
bigramas_elegiveis = {s: bigramas_doc[s][bigramas_doc[s]["frequencia"] >= FREQ_MINIMA_BIGRAMA] for s in SIGLAS}
for s in SIGLAS:
    relatar_empates(bigramas_elegiveis[s], "frequencia", TOP_N_BIGRAMAS_CONVERGENCIA, f"bigrama de {s}")
recortes_bigramas = {s: bigramas_elegiveis[s].head(TOP_N_BIGRAMAS_CONVERGENCIA).index for s in SIGLAS}
conv_bigramas = preparar_convergencia(bigramas_doc[SIG_A], bigramas_doc[SIG_B], recortes_bigramas,
                                      DESTAQUES_BIGRAMAS, rotulo_por_forma=True)

fig = grafico_convergencia(
    conv_bigramas,
    subtitulo=(f"Bigramas: as {TOP_N_BIGRAMAS_CONVERGENCIA} expressões de duas palavras mais frequentes de cada "
               f"documento (união: {len(conv_bigramas)} bigramas)"),
    descricao_ponto="bigrama (expressão de duas palavras de conteúdo)",
)
salvar_figura(fig, "06_convergencia_bigramas")
plt.show()

tabela_bigramas_conv = tabela_convergencia_exibicao(conv_bigramas, "bigrama (lemas)")
salvar_tabela(tabela_bigramas_conv, "06_convergencia_bigramas")
exibir(tabela_bigramas_conv)
sintese_convergencia(conv_bigramas, "bigramas", TOP_N_BIGRAMAS_CONVERGENCIA)

**Método e escolhas.** Mesmo desenho e mesmas regras da seção 16 (eixos, zonas, cores, faixa neutra, termos fora da escala, pontos sobrepostos e rótulos), aplicados às expressões de duas palavras. Os bigramas são os mesmos do item 14 dos notebooks individuais: pares de tokens de conteúdo **adjacentes no texto original** (na mesma frase e sem pontuação entre eles), agrupados pelo lema. Entram os 20 mais frequentes de cada documento (`TOP_N_BIGRAMAS_CONVERGENCIA`) entre os que têm frequência mínima `FREQ_MINIMA_BIGRAMA` (= 2), reunidos em uma única lista. O rótulo é a forma escrita mais frequente nos dois documentos (ex.: *developing countries* para o par de lemas *develop country*). X e Y são ocorrências por 1.000 tokens, com o mesmo denominador da seção 16 (total de tokens de cada documento). Os destaques pedidos pelo pesquisador (`DESTAQUES_BIGRAMAS`) são localizados pelo par de lemas ou pela forma escrita, sem diferenciar hífen (*foster multistakeholder* = *foster multi-stakeholder*); "global IA", no pedido, corresponde a *global ai* no texto. A tabela traz os valores exatos, o *Log Ratio* e o G² de cada bigrama.

**Como ler.** Como na seção 16: quanto mais à direita, mais frequente a expressão no BRICS; quanto mais acima, mais frequente na OCDE. Expressões sobre um eixo aparecem em um só documento.

**Interpretação.** *[Pendente. A redigir somente após a execução do notebook com os JSONs definitivos da skill 01, exclusivamente a partir dos valores exibidos acima. Toda afirmação de diferença deve indicar o critério que a sustenta (skill 03, seção 4.3).]*

**Fragilidades e ajustes possíveis.** Bigramas têm frequências baixas, o que torna as taxas instáveis e faz muitos termos coincidirem no mesmo ponto. Quando há empate no ponto de corte, a escolha segue a ordem alfabética, como no item 14, e é informada acima do gráfico; com frequências baixas, esse critério pode decidir quais bigramas entram. A zona é descritiva e não é teste de significância (ver, na tabela, a coluna dos critérios da seção 14). Variações de lematização podem dividir as ocorrências de uma mesma expressão em mais de um par de lemas. Expressões com três ou mais palavras aparecem fragmentadas, e expressões interrompidas por *stopwords* (*rule of law*) não são captadas como bigramas.

## 18. Versão interativa (site)

*Construída por comando do pesquisador.*

**Link:** [Convergências e divergências no vocabulário da governança da IA — BRICS × OCDE](https://claude.ai/artifact/RHof34mb3rkKAZL44Uyksm)

Versão interativa dos gráficos das seções 16 (unigramas) e 17 (bigramas), com os mesmos dados, cores, zonas e escala inicial; os valores foram conferidos com as tabelas exibidas acima. No site, é possível passar o mouse sobre cada ponto para ver o termo, as formas escritas, as frequências por 1.000 tokens, a razão entre as taxas e a zona; aproximar e restaurar a visualização; buscar e selecionar termos; alternar entre unigramas e bigramas; e salvar a imagem do gráfico montado. A metodologia e as instruções de leitura estão no fim da página.

O link é privado até ser compartilhado pelo menu **Compartilhar** da própria página.